# Detector benchmark — run & compare

This notebook drives the `yolo_bench` library rather than shelling out to the CLI.
The two layers are deliberately separate:

* **inference tier** (`yolo_bench.Detector`, `archs/`, `runtimes/`) — reusable Python, importable by other projects.
* **harness tier** (`yolo_bench.orchestrate`, `yolo_bench.plots`, this notebook) — test orchestration and visualisation.

Two practical consequences:

1. Every configuration runs in **its own subprocess**. Running a whole matrix in one interpreter made
   `host_rss_mb` grow monotonically (1.2 → 4.5 GB) purely from accumulation, which silently invalidated
   the memory metric. Isolation costs a process spawn per config and buys trustworthy numbers.
2. Cells are independent, so you can run a cheap probe, then a bigger matrix, without re-running everything.

## 1. Environment

This repo gets cloned onto an **RTX 3070** and a **Jetson Orin Nano** as well as this 3090 box, and the
result CSVs are merged. So every row carries a full fingerprint: CPU model/cores/RAM, GPU name + compute
capability + architecture, OS/kernel, driver, and library versions (TensorRT, ONNX Runtime, OpenVINO, torch).

That matters because the machines differ in ways that change results:

* TensorRT **11.3** here vs **10.16** on the Orin (JetPack 7.2) — different engine builders.
* The CPU backends are dominated by the host CPU: desktop i7-6700K vs Orin's 6-core Cortex-A78AE.
* On Jetson, the **power mode** changes results more than the software stack does.

In [16]:
import sys, glob, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.io as pio

# 'notebook_connected' renders inline but loads plotly.js from a CDN, keeping the
# .ipynb small. Plain 'notebook' embeds all ~4.8 MB of plotly.js in every figure.
for _renderer in ("notebook_connected", "notebook", "iframe"):
    try:
        pio.renderers.default = _renderer
        break
    except Exception:
        continue

from yolo_bench import orchestrate, plots
from yolo_bench.env import human_summary

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

print(human_summary())

host      : server  [x86_64]
os        : Ubuntu 26.04.1 LTS (kernel 7.0.0-34-generic)
cpu       : Intel(R) Core(TM) i7-6700K CPU @ 4.00GHz  [4c/8t, 30.7 GB RAM]
gpu       : NVIDIA GeForce RTX 3090 (Ampere, SM 8.6, 23.6 GB)
driver    : 595.91.07
cuda/cudnn: 13.0 / 9.40.0
libs      : tensorrt 11.3.0.99, onnxruntime 1.30.0, openvino 2026.4.0, torch 2.14.0+cu130 (cuda)
run       : server-20260926T175132Z at 2026-09-26T19:51:32+0200


## 2. Choose an experiment

Named experiments live in `orchestrate.EXPERIMENTS` so a setup is defined once instead of being re-typed
in every cell. Each is a plain dataclass, so any field can be overridden at call time.

In [17]:
pd.DataFrame([
    {
        "experiment": name,
        "description": exp.description,
        "models": ", ".join(exp.models),
        "runtimes": ", ".join(exp.runtimes),
        "precisions": ", ".join(exp.precisions),
        "frames": exp.frames,
    }
    for name, exp in orchestrate.EXPERIMENTS.items()
])

,experiment,description,models,runtimes,precisions,frames
0,smoke,"One model, one runtime — proves the stack runs...",yolo11s,tensorrt,"fp32, fp16",20
1,gpu,"All CUDA backends x both precisions, all models.","yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_trt, ort_cuda, pytorch, openvino","fp32, fp16",100
2,cpu,CPU baselines only (fp32: CPU EPs have no FP16...,"yolo11s, yolo11l, yolo26s, yolo26l","openvino_cpu, ort_cpu, pytorch_cpu",fp32,100
3,headline,"Everything: all runtimes, both precisions, all...","yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_trt, ort_cuda, pytorch, openvino...","fp32, fp16",100
4,precision,FP16 question only: native TRT vs ORT-CUDA vs ...,"yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_cuda, pytorch","fp32, fp16",100
5,small-vs-large,Same-architecture size scaling at the best-kno...,"yolo11s, yolo11l",tensorrt,fp16,100
6,yolo11-vs-yolo26,Head-rewrite question: yolo11 vs yolo26 at mat...,"yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_cuda",fp16,100


Which runtimes actually work here depends on the environment (for example ORT's TensorRT EP needs a
TensorRT major matching the ORT build). The registry knows, so ask it instead of guessing — unsupported
combinations are reported as **skipped** with a reason, never as a fake measurement.

In [18]:
print(__import__("yolo_bench.runtimes", fromlist=["describe"]).describe())

kind           group precisions   available label
tensorrt       cuda  fp32/fp16    yes       TensorRT (native)
ort_trt        cuda  fp32/fp16    yes       ONNX Runtime (TensorRT EP)
ort_cuda       cuda  fp32/fp16    yes       ONNX Runtime (CUDA EP)
pytorch        cuda  fp32/fp16    yes       PyTorch (CUDA)
openvino       cuda  fp32/fp16    yes       OpenVINO (GPU device)
openvino_cpu   cpu   fp32         yes       OpenVINO (CPU)
ort_cpu        cpu   fp32         yes       ONNX Runtime (CPU EP)
pytorch_cpu    cpu   fp32         yes       PyTorch (CPU)

  tensorrt       deserializes a .engine; where the Ampere tensor-core win shows
  ort_trt        needs a TensorRT major matching the ORT build (see README)
  ort_cuda       fp16 is accepted but gains nothing; read the fp32 row
  pytorch        eager; fp16 is *slower* here (latency-bound, see README)
  openvino       does NOT use TensorRT kernels; ~5x slower on NVIDIA
  openvino_cpu   fastest CPU backend measured here
  ort_cpu        CPU

## 3. Run

Start with `smoke` — one model, one runtime, a few frames. It proves the stack works end-to-end in a
couple of seconds before committing to a long run.

A full `headline` matrix (4 models × 8 runtimes × 2 precisions, 100 frames each) takes roughly 20 minutes
here. Results are written to `results/` so later cells read the CSV instead of re-running.

In [19]:
# Cheap end-to-end probe (~seconds). Override frames/warmup for a quick check.
# NOTE: this is a throwaway probe — it does NOT write to results/ (that dir holds
# only the canonical per-machine CSVs, the single source of truth).
smoke_rows = orchestrate.run_experiment("smoke", frames=20, warmup=5)
smoke_df = pd.DataFrame(smoke_rows)

cols = ["model", "runtime", "precision", "fps", "infer_ms", "e2e_ms",
        "peak_vram_gb", "runtime_version", "status"]
smoke_df[[c for c in cols if c in smoke_df.columns]].style.format(
    {"fps": "{:.1f}", "infer_ms": "{:.2f}", "e2e_ms": "{:.2f}", "peak_vram_gb": "{:.2f}"}
)

[orchestrate] FAIL yolo11s   tensorrt      fp32  AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
The CUDA driver logged these messages, which may provide useful details:
Returning 2 (CUDA_ERROR_OUT_OF_MEMORY) from cuDevicePrimaryCtxRetain


[orchestrate] FAIL yolo11s   tensorrt      fp16  AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
The CUDA driver logged these messages, which may provide useful details:
Returning 2 (CUDA_ERROR_OUT_OF_MEMORY) from cuDevicePrimaryCtxRetain




,model,runtime,precision,fps,infer_ms,e2e_ms,peak_vram_gb,runtime_version,status
0,yolo11s,tensorrt,fp32,nan,nan,nan,nan,None,failed
1,yolo11s,tensorrt,fp16,nan,nan,nan,nan,None,failed


In [20]:
# Single source of truth: one canonical CSV per machine in results/, named by
# machine tag (e.g. rtx3090.csv, orin-nano.csv). The filename stem IS the source
# tag, so adding a machine (e.g. rtx3070.csv) needs no code change here. Scratch
# runs (smoke, ad-hoc) are gitignored and never land in results/.
frames = []
for path in sorted(Path("results").glob("*.csv")):
    part = orchestrate.read_csv(path)
    part["source"] = path.stem          # tag == filename stem
    frames.append(part)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# Human label per machine, derived from the data (GPU name) so a new machine
# needs no edit here. Falls back to the tag if a row has no gpu_name.
MACHINE_LABEL = {}
for src in df["source"].unique():
    sub = df[df["source"] == src]
    gpu = sub["gpu_name"].dropna().unique()
    MACHINE_LABEL[src] = gpu[0] if len(gpu) else src

print(f"loaded {len(df)} rows from {len(frames)} machine(s): "
      f"{', '.join(MACHINE_LABEL[s] for s in df['source'].unique())}")
df[df["status"] == "ok"][["source", "model", "runtime", "precision", "fps",
                          "infer_ms", "e2e_ms", "peak_vram_gb",
                          "runtime_version", "status"]].head(12)

loaded 112 rows from 2 machine(s): Orin, NVIDIA GeForce RTX 3090


,source,model,runtime,precision,fps,infer_ms,e2e_ms,peak_vram_gb,runtime_version,status
0,orin-nano,yolo11s,tensorrt,fp32,31.577081,19.205451,31.668538,2.859276,10.16.2.10,ok
2,orin-nano,yolo11s,ort_cuda,fp32,31.304012,23.230854,31.944787,3.592255,1.23.0,ok
3,orin-nano,yolo11s,pytorch,fp32,22.527931,35.224319,44.389340,2.900520,2.13.0+cu132,ok
4,orin-nano,yolo11s,ort_cpu,fp32,4.195327,230.773009,238.360453,2.728168,1.23.0,ok
5,orin-nano,yolo11s,tensorrt,fp16,52.598229,10.263754,19.012047,2.745144,10.16.2.10,ok
7,orin-nano,yolo11s,ort_cuda,fp16,31.903420,23.046011,31.344602,3.581966,1.23.0,ok
8,orin-nano,yolo11s,pytorch,fp16,21.024561,38.620510,47.563418,3.121887,2.13.0+cu132,ok
9,orin-nano,yolo11l,tensorrt,fp32,18.824014,40.443528,53.123633,2.757912,10.16.2.10,ok
11,orin-nano,yolo11l,ort_cuda,fp32,14.186703,60.491204,70.488540,3.908436,1.23.0,ok
12,orin-nano,yolo11l,pytorch,fp32,8.630720,106.972763,115.865187,3.207531,2.13.0+cu132,ok


### Machine under test

In [21]:
# One header per machine, so a saved figure is self-describing.
from IPython.display import Markdown

for src in df["source"].unique():
    path = Path("results") / f"{src}.csv"
    if path.exists():
        Markdown(f"### {MACHINE_LABEL.get(src, src)}")
        Markdown(plots.machine_header(orchestrate.read_csv(path)))

## 4. Q1 — Runtime comparison

*Which backend is fastest, and does the ranking hold as the model grows?*

A **grouped bar chart** (x = model, one bar per runtime) with a log y-axis so
CPU and CUDA backends share the same scale. Small and large models are in
separate panels because they sit on different ends of the throughput range.

Q1–Q3 are **per-machine** questions — the ranking and the FP16 gain differ
between the 3090 and the Orin, so each chart is shown once per machine. Only Q4
mixes machines, and only because that is the question it asks.

In [22]:
# FP16 is the production precision; FP32 is the reference. One chart per machine:
# the ranking is a property of the machine, not of the merged data.
for src in df["source"].unique():
    sub = df[df["source"] == src]
    plots.runtime_slope(sub, precision="fp16", title=MACHINE_LABEL.get(src, src)).show()

The **ranking heatmap** is the same data as one glance: colour answers "which
runtime is fast for this model" without six separate bar groups. Blank cells are
combinations that were skipped (e.g. OpenVINO not installed on the Orin), never
a zero. Log-scaled colour, because a linear scale would flatten the CPU rows to
one shade.

In [23]:
for src in df["source"].unique():
    sub = df[df["source"] == src]
    plots.runtime_ranking_heatmap(sub, precision="fp16", title=MACHINE_LABEL.get(src, src)).show()

## 5. Q2 — Performance vs model cost

*How much throughput do you get per unit of model complexity?*

This is a **scatter**, because both axes are quantitative (GFLOPs on x, fps on y)
— bars force one axis categorical, which is the wrong shape here. The useful
reading is the *Pareto frontier*: points on the upper-left edge give the best
throughput per GFLOP. Colour is the model, so the yolo11-vs-yolo26 comparison at
matched size is visible within a panel.

In [24]:
# Throughput vs model complexity (GFLOPs). Faceted by size class, per machine.
for src in df["source"].unique():
    sub = df[df["source"] == src]
    plots.quality_vs_speed(sub, quality="gflops", precision="fp16", title=MACHINE_LABEL.get(src, src)).show()

## 6. Q3 — Quantization advantage

*What did FP16 actually buy, per runtime and model?*

A **heatmap** of the `fp16 / fp32` fps ratio using the same sequential (Viridis)
scale as the ranking heatmap: darker = bigger gain. A diverging scale centred on
1.0 would be misleading here because 1.0 is not the middle of the observed range
(~0.5–2.0), so it paints every "no gain" cell reddish. The annotated number is
always the true, unclamped value.

Shown per machine: the answer is machine-specific. On the 3090 (Ampere tensor
cores) TRT's FP16 gain is modest; on the Orin it is much larger — averaging the
two would hide exactly that difference.

In [25]:
# FP16 speedup per (runtime, model). White ≈ 1.0x = FP16 bought nothing.
# Per machine: the gain is machine-specific (e.g. TRT on the Orin gains far more
# than TRT on the 3090), so merging would average two different answers.
for src in df["source"].unique():
    sub = df[df["source"] == src]
    plots.quantization_gain_heatmap(sub, metric="fps", title=MACHINE_LABEL.get(src, src)).show()

## 7. Q4 — GPU comparison

*How does the same measurement move across machines?*

**Grouped bars**, one per (machine, runtime): the first machine at full opacity,
later machines lighter, so a pair of bars at one model reads as "same runtime,
two machines". The interesting output is the *gap between machines* at each
model size. Every row carries `gpu_name`/`gpu_arch`/`cpu` and the library
versions, which is what makes this possible — the legend shows the runtime
version so a mixed TensorRT-major comparison (3090 TRT 11.3 vs Orin TRT 10.16)
is *visible*, not implied.

Caveats when reading a merged table:

* Engines are **not portable** — each machine builds its own `.engine`.
* The Orin runs TensorRT 10.16, so its `tensorrt` rows are not directly comparable to the 3090's TRT 11.3 rows.
* On the Orin, the **power mode** (`nvpmodel`) dominates the result — this run was MAXN_SUPER.

In [26]:
# The cross-machine chart. `df` already carries both machines (see the load cell).
ok = df[df["status"] == "ok"]
print(f"{ok['source'].nunique()} machine(s), {len(ok)} successful rows")
plots.gpu_comparison(df, precision="fp16").show()

2 machine(s), 72 successful rows


In [27]:
# Same model/runtime across machines — the actual portability question.
ok = df[df["status"] == "ok"]
pivot = ok.pivot_table(
    index=["model", "runtime", "precision"],
    columns="gpu_name", values="fps", aggfunc="median",
)
pivot.round(1)

gpu_name                        NVIDIA GeForce RTX 3090  Orin
model   runtime      precision                               
yolo11l openvino     fp16                           9.4   NaN
                     fp32                           9.3   NaN
        openvino_cpu fp32                           3.5   NaN
        ort_cpu      fp32                           3.6   1.2
        ort_cuda     fp16                          78.5  14.1
                     fp32                          77.0  14.2
        pytorch      fp16                          44.5  12.2
                     fp32                          50.4   8.6
        pytorch_cpu  fp32                           2.5   NaN
        tensorrt     fp16                         136.7  31.0
                     fp32                          84.5  18.8
yolo11s openvino     fp16                          32.5   NaN
                     fp32                          32.1   NaN
        openvino_cpu fp32                          13.0   NaN
        ort_cpu      fp32                           8.4   4.2
        ort_cuda     fp16                         135.4  31.9
                     fp32                         129.8  31.3
        pytorch      fp16                          71.1  21.0
                     fp32                          82.5  22.5
        pytorch_cpu  fp32                           7.8   NaN
        tensorrt     fp16                         171.5  52.6
                     fp32                         138.3  31.6
yolo26l openvino     fp16                           9.4   NaN
                     fp32                           9.3   NaN
        openvino_cpu fp32                           3.5   NaN
        ort_cpu      fp32                           2.4   1.3
        ort_cuda     fp16                          79.9  14.3
                     fp32                          78.9  14.1
        pytorch      fp16                          41.2  11.7
                     fp32                          50.0   8.1
        pytorch_cpu  fp32                           2.4   NaN
        tensorrt     fp16                         149.3  30.9
                     fp32                          84.7  19.0
yolo26s openvino     fp16                          32.7   NaN
                     fp32                          32.4   NaN
        openvino_cpu fp32                          13.2   NaN
        ort_cpu      fp32                           9.2   4.3
        ort_cuda     fp16                         130.6  31.4
                     fp32                         127.4  32.0
        pytorch      fp16                          56.4  16.7
                     fp32                          66.6  18.9
        pytorch_cpu  fp32                           7.0   NaN
        tensorrt     fp16                         167.7  55.1
                     fp32                         134.2  31.3

## 8. Insights

The raw numbers live **only** in the per-machine CSVs (`results/<tag>.csv`) —
that is the single source of truth. This cell derives a short markdown report of
the key findings and writes it to `results/insights.md` (gitignored: it is
regenerated from the CSVs, never hand-edited). No HTML is emitted; the charts
above are the figures.

In [28]:
# Derive a short markdown report from the canonical CSVs and write it to
# results/insights.md. This is *derived* output (regenerated on every run), so
# it is gitignored — the CSVs remain the single source of truth. No HTML.
from IPython.display import Markdown

def _md_table(frame: pd.DataFrame) -> str:
    """Render a DataFrame as a GitHub markdown table (no tabulate dependency).

    A non-default index (e.g. model, or a (model, runtime) MultiIndex) is
    promoted to leading column(s) so row labels are not lost.
    """
    if not isinstance(frame.index, pd.RangeIndex):
        frame = frame.reset_index()
    cols = list(frame.columns)
    head = "| " + " | ".join(str(c) for c in cols) + " |"
    sep  = "|---" * len(cols) + "|"
    body = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |"
            for row in frame.itertuples(index=False)]
    return "\n".join([head, sep, *body])

ok = df[df["status"] == "ok"].copy()
lines = ["# Benchmark insights", "",
         f"_Derived from {len(df['source'].unique())} machine(s): "
         f"{', '.join(MACHINE_LABEL[s] for s in df['source'].unique())}. "
         "Regenerated by `results.ipynb` — do not edit by hand._", ""]

# 1) Per machine: fastest runtime per model (fp16, the production precision).
lines += ["## Fastest runtime per model (fp16)", ""]
for src in df["source"].unique():
    sub = ok[(ok["source"] == src) & (ok["precision"] == "fp16")]
    best = (sub.loc[sub.groupby("model")["fps"].idxmax()]
                .set_index("model")[["runtime", "fps"]]
                .round(1))
    lines += [f"### {MACHINE_LABEL.get(src, src)}", "", _md_table(best), ""]

# 2) Per machine: FP16 gain for tensorrt (the runtime where it matters).
lines += ["## TensorRT FP16 / FP32 fps ratio", ""]
for src in df["source"].unique():
    sub = ok[ok["source"] == src]
    trt = sub[sub["runtime"] == "tensorrt"].pivot_table(
        index="model", columns="precision", values="fps", aggfunc="mean")
    if {"fp16", "fp32"}.issubset(trt.columns):
        trt["fp16/fp32"] = (trt["fp16"] / trt["fp32"]).round(2)
        trt[["fp16", "fp32"]] = trt[["fp16", "fp32"]].round(1)
        lines += [f"### {MACHINE_LABEL.get(src, src)}", "", _md_table(trt), ""]

# 3) Cross-machine: fp16 fps by (model, runtime) across machines.
lines += ["## Cross-machine fps (fp16)", "",
          "_Same model+runtime on different hardware. TensorRT rows are not "
          "version-comparable (TRT 11 vs 10) — see README._", ""]
cross = (ok[ok["precision"] == "fp16"]
         .pivot_table(index=["model", "runtime"], columns="source",
                      values="fps", aggfunc="median").round(1))
lines += [_md_table(cross), ""]

out = Path("results/insights.md")
out.write_text("\n".join(lines))
print(f"wrote {out} ({len(lines)} lines)")
Markdown("\n".join(lines))

wrote results/insights.md (30 lines)


# Benchmark insights

_Derived from 2 machine(s): Orin, NVIDIA GeForce RTX 3090. Regenerated by `results.ipynb` — do not edit by hand._

## Fastest runtime per model (fp16)

### Orin

| model | runtime | fps |
|---|---|---|
| yolo11l | tensorrt | 31.0 |
| yolo11s | tensorrt | 52.6 |
| yolo26l | tensorrt | 30.9 |
| yolo26s | tensorrt | 55.1 |

### NVIDIA GeForce RTX 3090

| model | runtime | fps |
|---|---|---|
| yolo11l | tensorrt | 136.7 |
| yolo11s | tensorrt | 171.5 |
| yolo26l | tensorrt | 149.3 |
| yolo26s | tensorrt | 167.7 |

## TensorRT FP16 / FP32 fps ratio

### Orin

| model | fp16 | fp32 | fp16/fp32 |
|---|---|---|---|
| yolo11l | 31.0 | 18.8 | 1.65 |
| yolo11s | 52.6 | 31.6 | 1.67 |
| yolo26l | 30.9 | 19.0 | 1.62 |
| yolo26s | 55.1 | 31.3 | 1.76 |

### NVIDIA GeForce RTX 3090

| model | fp16 | fp32 | fp16/fp32 |
|---|---|---|---|
| yolo11l | 136.7 | 84.5 | 1.62 |
| yolo11s | 171.5 | 138.3 | 1.24 |
| yolo26l | 149.3 | 84.7 | 1.76 |
| yolo26s | 167.7 | 134.2 | 1.25 |

## Cross-machine fps (fp16)

_Same model+runtime on different hardware. TensorRT rows are not version-comparable (TRT 11 vs 10) — see README._

| model | runtime | orin-nano | rtx3090 |
|---|---|---|---|
| yolo11l | openvino |  | 9.4 |
| yolo11l | ort_cuda | 14.1 | 78.5 |
| yolo11l | pytorch | 12.2 | 44.5 |
| yolo11l | tensorrt | 31.0 | 136.7 |
| yolo11s | openvino |  | 32.5 |
| yolo11s | ort_cuda | 31.9 | 135.4 |
| yolo11s | pytorch | 21.0 | 71.1 |
| yolo11s | tensorrt | 52.6 | 171.5 |
| yolo26l | openvino |  | 9.4 |
| yolo26l | ort_cuda | 14.3 | 79.9 |
| yolo26l | pytorch | 11.7 | 41.2 |
| yolo26l | tensorrt | 30.9 | 149.3 |
| yolo26s | openvino |  | 32.7 |
| yolo26s | ort_cuda | 31.4 | 130.6 |
| yolo26s | pytorch | 16.7 | 56.4 |
| yolo26s | tensorrt | 55.1 | 167.7 |
